In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window


def transformSales_order_detail(SalesOrderDetail_df):

    windowSpec_rw = Window.partitionBy("SalesOrderDetailID").orderBy("ModifiedDate")
    SalesOrderDetail_df = SalesOrderDetail_df.filter(F.col("OrderQty") >= 1).withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    SalesOrderDetail_df = SalesOrderDetail_df.filter(F.col("rw") == 1).drop("rw")
    SalesOrderDetail_df = SalesOrderDetail_df.withColumn("processed_timestamp", F.current_timestamp())
    SalesOrderDetail_df = SalesOrderDetail_df.select( 
       F.col("SalesOrderID").cast(IntegerType()).alias("SalesOrderID"),
       F.col("SalesOrderDetailID").cast(IntegerType()).alias("SalesOrderDetailID"),
       F.trim(F.col("CarrierTrackingNumber")).alias("CarrierTrackingNumber"),
       F.col("OrderQty").cast(IntegerType()).alias("OrderQty"),
       F.col("ProductID").cast(IntegerType()).alias("ProductID"),
       F.col("SpecialOfferID").cast(IntegerType()).alias("SpecialOfferID"),
       F.col("UnitPrice").cast(DecimalType(19, 4)).alias("UnitPrice"),
       F.col("UnitPriceDiscount").cast(DecimalType(19, 4)).alias("UnitPriceDiscount"),
       F.col("LineTotal").cast(DecimalType(19, 4)).alias("LineTotal"),
       F.col("rowguid").cast(StringType()).alias("rowguid"),
       F.col("ModifiedDate").cast(DateType()) .alias("ModifiedDate"),
       F.col("_rescued_data").cast(StringType()).alias("_rescued_data"),
       F.col("processed_timestamp")
    )
                                 
    return SalesOrderDetail_df




if __name__ == "__main__":

    SalesOrderDetail_tbl = dbutils.widgets.get("SalesOrderDetail")
    SalesOrderDetail_df = df = spark.read.table(SalesOrderDetail_tbl)
    SalesOrderDetail_df_tgt = transformSales_order_Detail(SalesOrderDetail_df)
    display(SalesOrderDetail_df_tgt)